# auto_packet_analyze_v3 — Kaggle 파이프라인

pcap 하나(또는 여러 개)를 넣으면 **extract_log → evidence → LLM 분석 → 차단정책(Suricata) → 한글 보고서(md)** 까지 자동으로 돈다.
사람은 마지막에 **보고서를 읽고 차단 적용 여부(o/x)만** 정한다 — "인간 개입의 최소화".

## 실행 전 준비 (필수)
1. 우측 **Settings → Internet: ON** (apt/ollama/모델 다운로드에 필요)
2. 우측 **Settings → Accelerator: GPU** (gemma4:26b 는 GPU 필요)
3. **Add Input → 업로드한 pcap 데이터셋 연결** (pcap 은 `/kaggle/input/<dataset>/` 에 놓임)

## 실행 방법
- **최초 실행**: Run All. (setup + 모델 pull — 오래 걸림. 세션이 휘발성이라 새 세션마다 반복)
- **코드 수정 후 재실행**: 아래 **"4. ★ 재실행 시작점"** 셀부터 실행.
  git pull 만 하고 모델/패키지 재설치 없이 pcap 분석~보고서까지 다시 돈다.
- pcap 별로 **extract → evidence(로그로 숨김) → run.py(분석) → make_policy(룰) → render_report(md)** 순으로 돈다.
- 산출물: `reports/<name>.json`(분석) · `reports/<name>.rules`(Suricata 차단정책) · `reports/<name>.md`(최종 한글 보고서).

> **모델 변경**: 코드가 아니라 레포 루트 `.env` 의 `MODEL=` 한 줄만 바꾼다.
> 이 노트북은 MODEL 을 `.env` 에서 읽어 setup.sh(모델 pull)에 넘기고,
> run.py / render_report 에는 MODEL 을 강제하지 않아 config.py 가 `.env` 를 그대로 읽게 둔다.

## 1. 설정 + 레포 clone (최초 1회)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/DAADAISMYLIFE/auto_packet_analyze_v3.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/auto_packet_analyze_v3"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)

def env_model(repo, default="gemma4:26b"):
    """레포 루트 .env 의 MODEL= 값 (config.py 와 같은 단일 소스). 없으면 default."""
    envf = os.path.join(repo, ".env")
    if os.path.exists(envf):
        for line in open(envf, encoding="utf-8"):
            line = line.strip()
            if line.startswith("MODEL=") and not line.startswith("#"):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return default

MODEL = env_model(REPO)          # setup.sh 의 모델 pull 용 (run.py 는 config.py→.env 로 읽음)
print("repo :", REPO)
print("model:", MODEL, "(.env 기준)")

## 2. 환경 구성 (최초 1회 — suricata/zeek/ollama 설치 + 모델 pull)
`setup.sh` 가 전부 처리한다. **오래 걸림.** 재실행 시에는 이 셀을 건너뛴다.

In [ ]:
# Kaggle 은 root 라 sudo 없이 설치됨. .env 의 MODEL 을 pull + test.py 스모크까지 수행.
# {REPO}/{MODEL} 는 Jupyter 가 위 셀의 파이썬 변수로 치환한다.
!cd {REPO} && MODEL={MODEL} bash setup.sh

## 3. 파이썬 패키지 (최초 1회)

In [ ]:
!pip install -q ollama   # run.py 가 쓰는 유일한 외부 패키지 (config.py 는 표준 라이브러리만)

## 4. ★ 재실행 시작점 — git pull + 전체 pcap 분석

코드/프롬프트/`.env` 를 고치고 다시 돌릴 때는 **이 셀부터** 실행하면 된다.
- `git pull` 로 코드만 최신화 (모델/패키지 재설치 없음)
- ollama 서버가 죽어 있으면 다시 띄운다 (모델 다운로드는 다시 안 함)
- **MODEL 은 run.py 에 강제하지 않는다** → config.py 가 `.env` 를 읽는다 (단일 소스)
- pcap 별로: extract_log + build_evidence 는 **로그 파일로 숨기고**, `run.py` 출력만 표시

In [ ]:
import os, glob, subprocess, time, urllib.request

# ── 셀 단독 실행에도 안전하도록 변수 재정의 (설치는 하지 않음) ──
WORK = "/kaggle/working"
REPO = f"{WORK}/auto_packet_analyze_v3"
assert os.path.isdir(REPO), "레포가 없음 — 최초 실행이면 셀 1~3 부터 실행할 것"
os.chdir(REPO)

# ── 코드/프롬프트/.env 최신화 (fast-forward 만; 재설치 없음) ──
subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=False)

def env_model(repo, default="gemma4:26b"):
    envf = os.path.join(repo, ".env")
    if os.path.exists(envf):
        for line in open(envf, encoding="utf-8"):
            line = line.strip()
            if line.startswith("MODEL=") and not line.startswith("#"):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return default
print("model (.env):", env_model(REPO))

# ── ollama 서버 보장 (커널 재시작으로 죽었으면 재기동; 모델은 디스크에 있음) ──
def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
        return True
    except Exception:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
print("ollama up:", ollama_up())

# run.py / render_report 서브프로세스 env: MODEL 제거 → config.py 가 .env 를 단일 소스로 읽음
run_env = {k: v for k, v in os.environ.items() if k != "MODEL"}
LLM = os.path.join(REPO, "llm")

# ── 파이프라인: pcap 마다 extract → evidence(숨김) → run.py → make_policy → render_report ──
pcaps = sorted(glob.glob("/kaggle/input/**/*.pcap", recursive=True)
               + glob.glob("/kaggle/input/**/*.pcapng", recursive=True))
print("발견된 pcap:", pcaps or "(없음 — Add Input 으로 데이터셋 연결했는지 확인)")

for p in pcaps:
    name = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 60 + f"\n  {name}\n" + "=" * 60)

    # 4-1) 추출 + evidence — 출력은 로그 파일로 (보고 싶으면 파일 열기)
    prep_log = f"{WORK}/prep_{name}.log"
    with open(prep_log, "w") as lf:
        subprocess.run(["bash", "scripts/extract_log.sh", p], cwd=REPO,
                       stdout=lf, stderr=subprocess.STDOUT, check=False)
        subprocess.run(["python", "scripts/build_evidence.py", name], cwd=REPO,
                       stdout=lf, stderr=subprocess.STDOUT, check=False)
    print(f"(준비 완료 — extract/evidence 로그: {prep_log})")

    # 4-2) LLM 분석 — run.py 출력만 표시. llm/ 안에서 실행(from tools/config import ...)
    subprocess.run(["python", "run.py", name], cwd=LLM, env=run_env, check=False)

    # 4-3) 차단정책 생성 (순수 코드, chat 없음) → reports/<name>.rules + suricata -T 검증
    subprocess.run(["python", "scripts/make_policy.py", name, "--validate"],
                   cwd=REPO, check=False)

    # 4-4) 한글 MD 보고서 (코드가 표/타임라인/룰 주입 + LLM 서술 1콜) → reports/<name>.md
    subprocess.run(["python", "render_report.py", name], cwd=LLM, env=run_env, check=False)

## 5. 보고서 + 채점
`run.py` 가 저장한 `reports/<name>.json` 을 보여주고, `scripts/score.py` 로 `answers/truth` 와 비교해 지표를 낸다.
(truth 가 있는 케이스만 채점 — 신원/IOC/해시 recall, grounding=환각탐지, infra오인, verdict)

In [ ]:
import glob, os, json, subprocess
from IPython.display import Markdown, display

# ── 채점: reports/*.json 을 answers/truth 와 비교 (표는 stdout 으로) ──
print("="*60)
subprocess.run(["python", "scripts/score.py", "reports"], cwd=REPO, check=False)
print("="*60)

# ── 보고서 JSON 표시 ──
reports = sorted(glob.glob(f"{REPO}/reports/*.json"))
if not reports:
    print("(보고서 없음 — run.py 출력은 위 파이프라인 셀 stdout 확인)")
for rp in reports:
    rep = json.load(open(rp, encoding="utf-8"))
    a = rep.get("analysis") or {}
    display(Markdown(f"---\n### 📄 {os.path.basename(rp)} — verdict: **{rep.get('verdict')}**"))
    display(Markdown("```json\n" + json.dumps(rep, ensure_ascii=False, indent=2) + "\n```"))


## 6. 최종 보고서 (한글 md) + 차단정책

`render_report.py` 가 만든 `reports/<name>.md` 를 그대로 렌더링한다.
피해자/IOC/타임라인/차단룰은 **코드가 표로 주입**(재오염 없음), 개요·시나리오·권고 서술만 **LLM 한글**.
보고서 끝의 **`[ o / x ]`** 가 사람이 차단정책 적용 여부를 정하는 지점이다.

In [ ]:
import glob, os
from IPython.display import Markdown, display

# ── 최종 산출물: 한글 보고서(.md) + 차단정책(.rules) ──
mds = sorted(glob.glob(f"{REPO}/reports/*.md"))
if not mds:
    print("(md 보고서 없음 — 위 파이프라인 셀이 render_report 까지 돌았는지 확인)")
for md in mds:
    name = os.path.splitext(os.path.basename(md))[0]
    rules = f"{REPO}/reports/{name}.rules"
    display(Markdown(open(md, encoding="utf-8").read()))
    if os.path.exists(rules):
        display(Markdown(
            f"<details><summary>차단정책 원본 파일 — <code>{name}.rules</code></summary>\n\n```\n"
            + open(rules, encoding="utf-8").read() + "```\n</details>"))
    display(Markdown("\n---\n"))